In [1]:
from pathlib import Path
import pandas as pd

arquivo = Path(
    "../reports/attachments/"
    "TL-A243-2-v4 State Firearm Law Database 6.0.xlsx"
)

excel = pd.ExcelFile(arquivo)

print("Abas encontradas:")
for aba in excel.sheet_names:
    print("-", aba)

Abas encontradas:
- AboutThisFile
- Database
- Change Summary


In [2]:
# Ler somente as 5 primeiras linhas
df_rand = pd.read_excel(
    arquivo,
    sheet_name="Database",
    nrows=5
)

print("Dimensões da amostra:", df_rand.shape)

print("\nNomes das colunas:")
print(df_rand.columns.tolist())

display(df_rand)

Dimensões da amostra: (5, 23)

Nomes das colunas:
['Law ID', 'State', 'State Postal Abbreviation', 'FIPS Code', 'Law Class (num)', 'Law Class', 'Law Class Subtype', 'Handguns or Long Guns', 'Effect', 'Type of Change', 'Effective Date Note', 'Effective Date Year', 'Effective Date Month', 'Effective Date Day', 'Statutory Citation', 'Content', 'Supersession Date Year', 'Supersession Date Month', 'Supersession Date Day', 'Age for Minimum Age Laws', 'Length of Waiting Period (days, handguns)', 'Additional Context and Notes', 'Discrepancies']


,Law ID,State,State Postal Abbreviation,FIPS Code,Law Class (num),Law Class,Law Class Subtype,Handguns or Long Guns,Effect,Type of Change,...,Effective Date Day,Statutory Citation,Content,Supersession Date Year,Supersession Date Month,Supersession Date Day,Age for Minimum Age Laws,"Length of Waiting Period (days, handguns)",Additional Context and Notes,Discrepancies
0,AK1002,Alaska,AK,2,2,carrying a concealed weapon (ccw),prohibited,handgun,Restrictive,Implement,...,1,Alaska Stat. § 11.55.010,It is unlawful for a person to carry concealed...,1994.0,10.0,1.0,NaN,NaN,Prior law prohibiting concealed carry enacted ...,NaN
1,AK1003,Alaska,AK,2,2,carrying a concealed weapon (ccw),shall issue,handgun,Permissive,Modify,...,1,Alaska Stat. § 18.65.700 et seq.,Sec. 18.65.700. Permit to carry a concealed ha...,2003.0,9.0,9.0,NaN,NaN,NaN,NaN
2,AK1004,Alaska,AK,2,2,carrying a concealed weapon (ccw),shall issue (permit not required),handgun,Permissive,Modify,...,9,Permit-less CCW allowed through change to Alas...,§ 11.61.220(a) A person commits the crime of m...,NaN,NaN,NaN,NaN,NaN,Permitting system maintained for residents see...,NaN
3,AK1005,Alaska,AK,2,3,castle doctrine,traditional,handgun and long gun,Permissive,Modify,...,1,Alaska Stat. § 11.81.335,b) A person may not use deadly force under thi...,2006.0,9.0,13.0,NaN,NaN,See 2006 S.B. No. 200 Ch. 68.,NaN
4,AK1006,Alaska,AK,2,3,castle doctrine,expanded 2,handgun and long gun,Permissive,Modify,...,13,Ak. Stat. Ann. § 11.81.335,A person may not use deadly force under this s...,2013.0,9.0,18.0,NaN,NaN,Catagorized as expanded 2 because removes duty...,NaN


In [3]:
# Carregar a aba completa
leis = pd.read_excel(
    arquivo,
    sheet_name="Database"
)

print("Total de registros:", len(leis))
print("Estados encontrados:", leis["State"].nunique())

print("\nCategorias de legislação:")
display(
    leis["Law Class"]
    .value_counts()
    .to_frame("Quantidade")
)

Total de registros: 1902
Estados encontrados: 51

Categorias de legislação:


,Quantidade
Law Class,
minimum age,378
background checks,265
prohibited possessor,243
firearm sales restrictions,159
carrying a concealed weapon (ccw),142
castle doctrine,123
waiting period,111
firearms in college/university,81
local laws preempted by state,64


In [4]:
categorias = [
    "carrying a concealed weapon (ccw)",
    "background checks",
    "castle doctrine"
]

selecionadas = leis[
    leis["Law Class"].isin(categorias)
]

resumo = (
    selecionadas
    .groupby(["Law Class", "Law Class Subtype"])
    .size()
    .reset_index(name="Registros")
    .sort_values(["Law Class", "Registros"], ascending=[True, False])
)

display(resumo)

,Law Class,Law Class Subtype,Registros
5,background checks,sales from dealer : federal,107
4,background checks,sales from dealer,60
2,background checks,private sales : point of sale,46
1,background checks,permit to purchase,28
0,background checks,extra time,23
3,background checks,private sales : point of sale (limited),1
8,carrying a concealed weapon (ccw),shall issue,56
6,carrying a concealed weapon (ccw),may issue,35
9,carrying a concealed weapon (ccw),shall issue (permit not required),29
7,carrying a concealed weapon (ccw),prohibited,22


In [5]:
colunas = [
    "State",
    "Law Class",
    "Law Class Subtype",
    "Effect",
    "Type of Change",
    "Effective Date Year",
    "Supersession Date Year"
]

display(selecionadas[colunas].head(15))

print("\nTipos de alteração:")
display(selecionadas["Type of Change"].value_counts())

print("\nDatas de substituição ausentes:")
print(selecionadas["Supersession Date Year"].isna().sum())

,State,Law Class,Law Class Subtype,Effect,Type of Change,Effective Date Year,Supersession Date Year
0,Alaska,carrying a concealed weapon (ccw),prohibited,Restrictive,Implement,1978,1994.0
1,Alaska,carrying a concealed weapon (ccw),shall issue,Permissive,Modify,1994,2003.0
2,Alaska,carrying a concealed weapon (ccw),shall issue (permit not required),Permissive,Modify,2003,NaN
3,Alaska,castle doctrine,traditional,Permissive,Modify,1978,2006.0
4,Alaska,castle doctrine,expanded 2,Permissive,Modify,2006,2013.0
10,Alaska,background checks,sales from dealer : federal,Restrictive,Implement,1994,1998.0
11,Alaska,background checks,sales from dealer : federal,Restrictive,Modify,1998,NaN
12,Alaska,castle doctrine,traditional,Permissive,Implement,1900,1978.0
17,Alaska,castle doctrine,stand your ground,Permissive,Modify,2013,NaN
24,Alabama,carrying a concealed weapon (ccw),shall issue,Permissive,Modify,1975,2023.0



Tipos de alteração:


Type of Change
Implement    297
Modify       218
Repeal        13
See note       2
Name: count, dtype: int64


Datas de substituição ausentes:
288


In [7]:
sem_substituicao = selecionadas[
    selecionadas["Supersession Date Year"].isna()
]

print("Registros sem substituição:", len(sem_substituicao))

display(
    sem_substituicao.groupby(
        ["Law Class", "Law Class Subtype"]
    ).size().reset_index(name="Quantidade")
)

Registros sem substituição: 288


,Law Class,Law Class Subtype,Quantidade
0,background checks,extra time,22
1,background checks,permit to purchase,20
2,background checks,private sales : point of sale,35
3,background checks,private sales : point of sale (limited),1
4,background checks,sales from dealer,49
5,background checks,sales from dealer : federal,54
6,carrying a concealed weapon (ccw),may issue,2
7,carrying a concealed weapon (ccw),shall issue,25
8,carrying a concealed weapon (ccw),shall issue (permit not required),29
9,castle doctrine,expanded 1,5


In [8]:
porte = leis[
    leis["Law Class"] == "carrying a concealed weapon (ccw)"
].copy()

colunas = [
    "State",
    "Law Class Subtype",
    "Type of Change",
    "Effective Date Year",
    "Supersession Date Year"
]

display(
    porte[porte["State"] == "Alaska"]
    [colunas]
    .sort_values("Effective Date Year")
)

,State,Law Class Subtype,Type of Change,Effective Date Year,Supersession Date Year
0,Alaska,prohibited,Implement,1978,1994.0
1,Alaska,shall issue,Modify,1994,2003.0
2,Alaska,shall issue (permit not required),Modify,2003,NaN


In [9]:
print("Registros de porte:", len(porte))

print("\nTipos de alteração:")
display(porte["Type of Change"].value_counts(dropna=False))

print("\nAnos de entrada em vigor:")
print(
    porte["Effective Date Year"].min(),
    "até",
    porte["Effective Date Year"].max()
)

print("\nRegistros sem data de substituição:")
print(porte["Supersession Date Year"].isna().sum())

Registros de porte: 142

Tipos de alteração:


Type of Change
Modify       84
Implement    57
Repeal        1
Name: count, dtype: int64


Anos de entrada em vigor:
1813 até 2024

Registros sem data de substituição:
56


In [10]:
atuais = porte[
    porte["Supersession Date Year"].isna()
].copy()

resumo = (
    atuais.groupby("State")
    .agg(
        registros=("Law Class Subtype", "size"),
        modalidades=("Law Class Subtype", "nunique")
    )
    .sort_values("registros", ascending=False)
)

print("Estados encontrados:", len(resumo))

print("\nEstados com mais de um registro:")
display(resumo[resumo["registros"] > 1])

Estados encontrados: 51

Estados com mais de um registro:


,registros,modalidades
State,,
Louisiana,2,2
Hawaii,2,1
California,2,1
Wisconsin,2,1
New Jersey,2,1


In [11]:
colunas = [
    "Law ID",
    "State",
    "Law Class Subtype",
    "Effect",
    "Type of Change",
    "Effective Date Year",
    "Supersession Date Year"
]

louisiana = porte[
    porte["State"] == "Louisiana"
].copy()

display(
    louisiana[colunas]
    .sort_values("Effective Date Year")
)

,Law ID,State,Law Class Subtype,Effect,Type of Change,Effective Date Year,Supersession Date Year
748,LA1004,Louisiana,prohibited,Restrictive,Implement,1855,1975.0
747,LA1003,Louisiana,may issue,Permissive,Modify,1975,1996.0
749,LA1005,Louisiana,shall issue,Permissive,Modify,1996,NaN
777,LA1063,Louisiana,shall issue (permit not required),Permissive,Modify,2024,NaN


In [12]:
ids = ["LA1005", "LA1063"]

colunas = [
    "Law ID",
    "Effective Date Year",
    "Effective Date Month",
    "Effective Date Day",
    "Law Class Subtype",
    "Additional Context and Notes",
    "Caveats and Ambiguities"
]

print("Colunas disponíveis:")
print([c for c in colunas if c in leis.columns])

display(
    leis.loc[
        leis["Law ID"].isin(ids),
        [c for c in colunas if c in leis.columns]
    ].T
)

Colunas disponíveis:
['Law ID', 'Effective Date Year', 'Effective Date Month', 'Effective Date Day', 'Law Class Subtype', 'Additional Context and Notes']


,749,777
Law ID,LA1005,LA1063
Effective Date Year,1996,2024
Effective Date Month,4,8
Effective Date Day,19,1
Law Class Subtype,shall issue,shall issue (permit not required)
Additional Context and Notes,While LA has allowed permitless carry since 20...,NaN


In [13]:
registro = leis.loc[
    leis["Law ID"] == "LA1005"
].iloc[0]

print("Registro:", registro["Law ID"])
print("Modalidade:", registro["Law Class Subtype"])

print("\nObservações completas:")
print(registro["Additional Context and Notes"])

Registro: LA1005
Modalidade: shall issue

Observações completas:
While LA has allowed permitless carry since 2024, it did not repeal the  law establishing a process for applying for a CCW license.


In [15]:
# Teste inicial: porte oculto na Louisiana, 2019–2024

anos = range(2019, 2025)

for ano in anos:
    vigentes = louisiana[
        (louisiana["Effective Date Year"] <= ano)
        & (
            louisiana["Supersession Date Year"].isna()
            | (louisiana["Supersession Date Year"] > ano)
        )
    ]

    modalidades = vigentes["Law Class Subtype"].tolist()

    print(f"{ano}: {modalidades}")

2019: ['shall issue']
2020: ['shall issue']
2021: ['shall issue']
2022: ['shall issue']
2023: ['shall issue']
2024: ['shall issue', 'shall issue (permit not required)']


In [16]:
registros = []

for ano in range(2019, 2025):
    vigentes = louisiana[
        (louisiana["Effective Date Year"] <= ano)
        & (
            louisiana["Supersession Date Year"].isna()
            | (louisiana["Supersession Date Year"] > ano)
        )
    ]

    modalidades = set(vigentes["Law Class Subtype"])

    registros.append({
        "state": "Louisiana",
        "year": ano,
        "ccw_shall_issue": int("shall issue" in modalidades),
        "ccw_permitless": int(
            "shall issue (permit not required)" in modalidades
        )
    })

df_louisiana = pd.DataFrame(registros)

display(df_louisiana)

,state,year,ccw_shall_issue,ccw_permitless
0,Louisiana,2019,1,0
1,Louisiana,2020,1,0
2,Louisiana,2021,1,0
3,Louisiana,2022,1,0
4,Louisiana,2023,1,0
5,Louisiana,2024,1,1


In [17]:
registros = []

for estado in porte["State"].unique():
    dados_estado = porte[porte["State"] == estado]

    for ano in range(2019, 2025):
        vigentes = dados_estado[
            (dados_estado["Effective Date Year"] <= ano)
            & (
                dados_estado["Supersession Date Year"].isna()
                | (dados_estado["Supersession Date Year"] > ano)
            )
        ]

        modalidades = set(vigentes["Law Class Subtype"])

        registros.append({
            "state": estado,
            "year": ano,
            "ccw_shall_issue": int("shall issue" in modalidades),
            "ccw_permitless": int(
                "shall issue (permit not required)" in modalidades
            ),
            "ccw_records": len(vigentes)
        })

df_ccw = pd.DataFrame(registros)

print("Total de registros:", len(df_ccw))
print("Estados encontrados:", df_ccw["state"].nunique())

display(df_ccw.head(10))

Total de registros: 306
Estados encontrados: 51


,state,year,ccw_shall_issue,ccw_permitless,ccw_records
0,Alaska,2019,0,1,1
1,Alaska,2020,0,1,1
2,Alaska,2021,0,1,1
3,Alaska,2022,0,1,1
4,Alaska,2023,0,1,1
5,Alaska,2024,0,1,1
6,Alabama,2019,1,0,1
7,Alabama,2020,1,0,1
8,Alabama,2021,1,0,1
9,Alabama,2022,1,0,1


In [18]:
print("Registros sem legislação CCW identificada:")
display(df_ccw[df_ccw["ccw_records"] == 0])

print("\nRegistros por ano:")
display(df_ccw.groupby("year")["state"].nunique())

print("\nDuplicações:")
print(df_ccw.duplicated(["state", "year"]).sum())

Registros sem legislação CCW identificada:


,state,year,ccw_shall_issue,ccw_permitless,ccw_records



Registros por ano:


year
2019    51
2020    51
2021    51
2022    51
2023    51
2024    51
Name: state, dtype: int64


Duplicações:
0


In [19]:
for modalidade, coluna in [
    ("may issue", "ccw_may_issue"),
    ("prohibited", "ccw_prohibited")
]:
    registros_coluna = []

    for _, linha in df_ccw.iterrows():
        vigentes = porte[
            (porte["State"] == linha["state"])
            & (porte["Effective Date Year"] <= linha["year"])
            & (
                porte["Supersession Date Year"].isna()
                | (porte["Supersession Date Year"] > linha["year"])
            )
        ]

        registros_coluna.append(
            int(modalidade in set(vigentes["Law Class Subtype"]))
        )

    df_ccw[coluna] = registros_coluna

# Remover Washington, D.C. para manter os 50 estados
df_ccw = df_ccw[df_ccw["state"] != "District of Columbia"].copy()

print("Dimensões:", df_ccw.shape)

display(df_ccw.head(10))

Dimensões: (300, 7)


,state,year,ccw_shall_issue,ccw_permitless,ccw_records,ccw_may_issue,ccw_prohibited
0,Alaska,2019,0,1,1,0,0
1,Alaska,2020,0,1,1,0,0
2,Alaska,2021,0,1,1,0,0
3,Alaska,2022,0,1,1,0,0
4,Alaska,2023,0,1,1,0,0
5,Alaska,2024,0,1,1,0,0
6,Alabama,2019,1,0,1,0,0
7,Alabama,2020,1,0,1,0,0
8,Alabama,2021,1,0,1,0,0
9,Alabama,2022,1,0,1,0,0


In [20]:
colunas_ccw = [
    "ccw_shall_issue",
    "ccw_permitless",
    "ccw_may_issue",
    "ccw_prohibited"
]

print("Estados:", df_ccw["state"].nunique())
print("Registros:", len(df_ccw))

print("\nRegistros sem nenhuma modalidade:")
print((df_ccw[colunas_ccw].sum(axis=1) == 0).sum())

print("\nRegistros com múltiplas modalidades:")
print((df_ccw[colunas_ccw].sum(axis=1) > 1).sum())

Estados: 50
Registros: 300

Registros sem nenhuma modalidade:
0

Registros com múltiplas modalidades:
1


In [21]:
from pathlib import Path

destino = Path("../data/processed/rand_ccw_2019_2024.csv")

colunas_exportacao = [
    "state",
    "year",
    "ccw_shall_issue",
    "ccw_permitless",
    "ccw_may_issue",
    "ccw_prohibited"
]

df_ccw[colunas_exportacao].to_csv(destino, index=False)

print("Arquivo salvo em:", destino.resolve())

Arquivo salvo em: /media/jandersongustavo/f8223ea0-bc74-41e8-8f66-650dbe31ee07/Arquivos_Janderson/Documentos/Projetos/us-gun-policy-ml/data/processed/rand_ccw_2019_2024.csv


In [22]:
background = leis[
    leis["Law Class"] == "background checks"
].copy()

print("Registros:", len(background))

display(
    background[
        [
            "Law Class Subtype",
            "Type of Change",
            "Effective Date Year",
            "Supersession Date Year"
        ]
    ].head(10)
)

Registros: 265


,Law Class Subtype,Type of Change,Effective Date Year,Supersession Date Year
10,sales from dealer : federal,Implement,1994,1998.0
11,sales from dealer : federal,Modify,1998,NaN
37,sales from dealer : federal,Implement,1994,1998.0
38,sales from dealer : federal,Modify,1998,NaN
58,sales from dealer : federal,Implement,1994,1997.0
59,sales from dealer : federal,Repeal,1997,1998.0
60,sales from dealer : federal,Modify,1998,NaN
61,sales from dealer : federal,Modify,1998,NaN
76,sales from dealer,Repeal,1994,1998.0
77,sales from dealer,Implement,1998,NaN


In [25]:
vendas_privadas = background[
    background["Law Class Subtype"] == "private sales : point of sale"
].copy()

print("Registros:", len(vendas_privadas))
print("Estados:", vendas_privadas["State"].nunique())

print("\nTipos de alteração:")
display(vendas_privadas["Type of Change"].value_counts(dropna=False))

print("\nExemplo de registros:")
display(
    vendas_privadas[
        [
            "State",
            "Effect",
            "Type of Change",
            "Effective Date Year",
            "Supersession Date Year"
        ]
    ].head(10)
)

Registros: 46
Estados: 24

Tipos de alteração:


Type of Change
Implement    36
Modify        6
Repeal        4
Name: count, dtype: int64


Exemplo de registros:


,State,Effect,Type of Change,Effective Date Year,Supersession Date Year
108,California,Restrictive,Implement,1991,NaN
132,California,Restrictive,Implement,1991,NaN
189,Colorado,Restrictive,Implement,2013,NaN
199,Colorado,Restrictive,Implement,2013,NaN
237,Connecticut,Restrictive,Implement,1994,NaN
271,Connecticut,Restrictive,Implement,2013,2014.0
356,Delaware,Restrictive,Implement,2013,NaN
367,Delaware,Restrictive,Implement,2013,NaN
618,Illinois,Restrictive,Implement,1968,2013.0
639,Illinois,Restrictive,Modify,2013,NaN


In [26]:
print("Total de registros:", len(vendas_privadas))

print("\nEstados com registros:")
print(vendas_privadas["State"].nunique())

print("\nTipos de alteração:")
display(
    vendas_privadas["Type of Change"]
    .value_counts(dropna=False)
)

print("\nRegistros sem data de substituição:")
print(
    vendas_privadas["Supersession Date Year"]
    .isna()
    .sum()
)

Total de registros: 46

Estados com registros:
24

Tipos de alteração:


Type of Change
Implement    36
Modify        6
Repeal        4
Name: count, dtype: int64


Registros sem data de substituição:
35


In [27]:
ativos = vendas_privadas[
    vendas_privadas["Supersession Date Year"].isna()
]

print("Estados com registros sem substituição:")
print(ativos["State"].nunique())

print("\nTipos de alteração de todos os registros:")
print(vendas_privadas["Type of Change"].value_counts())

print("\nEstados com múltiplos registros sem substituição:")
display(
    ativos["State"]
    .value_counts()
    .loc[lambda x: x > 1]
)

Estados com registros sem substituição:
23

Tipos de alteração de todos os registros:
Type of Change
Implement    36
Modify        6
Repeal        4
Name: count, dtype: int64

Estados com múltiplos registros sem substituição:


State
California      2
Colorado        2
Delaware        2
Illinois        2
New Jersey      2
New Mexico      2
Nevada          2
New York        2
Oregon          2
Rhode Island    2
Vermont         2
Washington      2
Name: count, dtype: int64

In [28]:
# Inspecionar os registros da California
california = ativos[
    ativos["State"] == "California"
]

print("Colunas disponíveis:")
print(ativos.columns.tolist())

display(california.T)

Colunas disponíveis:
['Law ID', 'State', 'State Postal Abbreviation', 'FIPS Code', 'Law Class (num)', 'Law Class', 'Law Class Subtype', 'Handguns or Long Guns', 'Effect', 'Type of Change', 'Effective Date Note', 'Effective Date Year', 'Effective Date Month', 'Effective Date Day', 'Statutory Citation', 'Content', 'Supersession Date Year', 'Supersession Date Month', 'Supersession Date Day', 'Age for Minimum Age Laws', 'Length of Waiting Period (days, handguns)', 'Additional Context and Notes', 'Discrepancies']


,108,132
Law ID,CA1001,CA1040
State,California,California
State Postal Abbreviation,CA,CA
FIPS Code,6,6
Law Class (num),1,1
Law Class,background checks,background checks
Law Class Subtype,private sales : point of sale,private sales : point of sale
Handguns or Long Guns,long gun,handgun
Effect,Restrictive,Restrictive
Type of Change,Implement,Implement


In [29]:
print("Tipos de alteração:")
display(
    vendas_privadas["Type of Change"]
    .value_counts(dropna=False)
)

print("\nTipos de armas registrados:")
display(
    vendas_privadas["Handguns or Long Guns"]
    .value_counts(dropna=False)
)

print("\nEstados distintos:")
print(vendas_privadas["State"].nunique())

Tipos de alteração:


Type of Change
Implement    36
Modify        6
Repeal        4
Name: count, dtype: int64


Tipos de armas registrados:


Handguns or Long Guns
handgun                 23
long gun                16
handgun and long gun     7
Name: count, dtype: int64


Estados distintos:
24


In [30]:
display(
    vendas_privadas["Type of Change"]
    .value_counts(dropna=False)
)

display(
    vendas_privadas["Handguns or Long Guns"]
    .value_counts(dropna=False)
)

Type of Change
Implement    36
Modify        6
Repeal        4
Name: count, dtype: int64

Handguns or Long Guns
handgun                 23
long gun                16
handgun and long gun     7
Name: count, dtype: int64

In [31]:
display(
    vendas_privadas["Type of Change"]
    .value_counts(dropna=False)
)

Type of Change
Implement    36
Modify        6
Repeal        4
Name: count, dtype: int64

In [32]:
revogacoes = vendas_privadas[
    vendas_privadas["Type of Change"] == "Repeal"
].copy()

colunas = [
    "Law ID",
    "State",
    "Handguns or Long Guns",
    "Effect",
    "Type of Change",
    "Effective Date Year",
    "Supersession Date Year"
]

display(
    revogacoes[colunas]
    .sort_values(["State", "Effective Date Year"])
)

,Law ID,State,Handguns or Long Guns,Effect,Type of Change,Effective Date Year,Supersession Date Year
695,IN1058,Indiana,handgun,Permissive,Repeal,1998,NaN
992,MO1004,Missouri,handgun,Permissive,Repeal,2007,NaN
1099,NC1059,North Carolina,handgun,Permissive,Repeal,2023,NaN
1600,TN1002,Tennessee,handgun,Permissive,Repeal,1998,NaN


In [33]:
nc = vendas_privadas[
    vendas_privadas["State"] == "North Carolina"
].copy()

colunas = [
    "Law ID",
    "Handguns or Long Guns",
    "Effect",
    "Type of Change",
    "Effective Date Year",
    "Supersession Date Year",
]

display(
    nc[colunas]
    .sort_values("Effective Date Year")
)

,Law ID,Handguns or Long Guns,Effect,Type of Change,Effective Date Year,Supersession Date Year
1092,NC1046,handgun,Restrictive,Implement,1987,2023.0
1099,NC1059,handgun,Permissive,Repeal,2023,NaN


In [34]:
registros_nc = []

for ano in range(2019, 2025):
    vigentes = nc[
        (nc["Effective Date Year"] <= ano)
        & (
            nc["Supersession Date Year"].isna()
            | (nc["Supersession Date Year"] > ano)
        )
        & (nc["Type of Change"] != "Repeal")
    ]

    registros_nc.append({
        "state": "North Carolina",
        "year": ano,
        "background_handgun": int(
            vigentes["Handguns or Long Guns"]
            .isin(["handgun", "handgun and long gun"])
            .any()
        )
    })

df_nc = pd.DataFrame(registros_nc)

display(df_nc)

,state,year,background_handgun
0,North Carolina,2019,1
1,North Carolina,2020,1
2,North Carolina,2021,1
3,North Carolina,2022,1
4,North Carolina,2023,0
5,North Carolina,2024,0


In [35]:
registros_background = []

# Usar os 50 estados já identificados no dataset CCW
for estado in sorted(df_ccw["state"].unique()):

    dados_estado = vendas_privadas[
        vendas_privadas["State"] == estado
    ]

    for ano in range(2019, 2025):

        vigentes = dados_estado[
            (dados_estado["Effective Date Year"] <= ano)
            & (
                dados_estado["Supersession Date Year"].isna()
                | (dados_estado["Supersession Date Year"] > ano)
            )
            & (dados_estado["Type of Change"] != "Repeal")
        ]

        tipos = set(vigentes["Handguns or Long Guns"])

        registros_background.append({
            "state": estado,
            "year": ano,
            "background_handgun": int(
                bool(tipos & {"handgun", "handgun and long gun"})
            ),
            "background_longgun": int(
                bool(tipos & {"long gun", "handgun and long gun"})
            )
        })

df_background = pd.DataFrame(registros_background)

print("Dimensões:", df_background.shape)

display(df_background.head(10))

Dimensões: (300, 4)


,state,year,background_handgun,background_longgun
0,Alabama,2019,0,0
1,Alabama,2020,0,0
2,Alabama,2021,0,0
3,Alabama,2022,0,0
4,Alabama,2023,0,0
5,Alabama,2024,0,0
6,Alaska,2019,0,0
7,Alaska,2020,0,0
8,Alaska,2021,0,0
9,Alaska,2022,0,0


In [36]:
print("Estados:", df_background["state"].nunique())
print("Registros:", len(df_background))

print("\nDuplicações:")
print(df_background.duplicated(["state", "year"]).sum())

print("\nNorth Carolina:")
display(
    df_background[
        df_background["state"] == "North Carolina"
    ]
)

print("\nEstados com exigências identificadas em 2024:")
display(
    df_background[df_background["year"] == 2024][
        ["background_handgun", "background_longgun"]
    ].sum()
)

Estados: 50
Registros: 300

Duplicações:
0

North Carolina:


,state,year,background_handgun,background_longgun
192,North Carolina,2019,1,0
193,North Carolina,2020,1,0
194,North Carolina,2021,1,0
195,North Carolina,2022,1,0
196,North Carolina,2023,0,0
197,North Carolina,2024,0,0



Estados com exigências identificadas em 2024:


background_handgun    18
background_longgun    17
dtype: int64

In [37]:
from pathlib import Path

destino = Path("../data/processed/rand_background_2019_2024.csv")

df_background.to_csv(destino, index=False)

print("Arquivo salvo em:", destino.resolve())

Arquivo salvo em: /media/jandersongustavo/f8223ea0-bc74-41e8-8f66-650dbe31ee07/Arquivos_Janderson/Documentos/Projetos/us-gun-policy-ml/data/processed/rand_background_2019_2024.csv


In [43]:
import pandas as pd

arquivo_rand = (
    "../reports/attachments/"
    "TL-A243-2-v4 State Firearm Law Database 6.0.xlsx"
)

rand = pd.read_excel(
    arquivo_rand,
    sheet_name="Database"
)

print("RAND carregada:", rand.shape)

RAND carregada: (1902, 23)


In [44]:
estados_permitless = [
    "Arkansas", "Iowa", "Montana", "Tennessee",
    "Texas", "Utah", "Georgia", "Indiana",
    "Ohio", "Alabama", "Florida", "Nebraska"
]

In [45]:
registros_permitless = rand[
    rand["State"].isin(estados_permitless)
    & (
        rand["Law Class Subtype"]
        == "shall issue (permit not required)"
    )
    & rand["Effective Date Year"].between(2020, 2023)
]

display(
    registros_permitless[
        [
            "Law ID",
            "State",
            "Law Class Subtype",
            "Type of Change",
            "Effective Date Year",
            "Effective Date Month",
            "Effective Date Day"
        ]
    ].sort_values(["Effective Date Year", "State"])
)

print("Registros encontrados:", len(registros_permitless))

,Law ID,State,Law Class Subtype,Type of Change,Effective Date Year,Effective Date Month,Effective Date Day
69,AR1052,Arkansas,shall issue (permit not required),Modify,2021,8,31
569,IA1061,Iowa,shall issue (permit not required),Modify,2021,7,1
1068,MT1053,Montana,shall issue (permit not required),Modify,2021,2,18
1628,TN1055,Tennessee,shall issue (permit not required),Modify,2021,7,1
1660,TX1051,Texas,shall issue (permit not required),Modify,2021,9,1
1700,UT1070,Utah,shall issue (permit not required),Modify,2021,5,5
465,GA1062,Georgia,shall issue (permit not required),Modify,2022,4,12
696,IN1059,Indiana,shall issue (permit not required),Modify,2022,7,1
1389,OH1051,Ohio,shall issue (permit not required),Modify,2022,6,13
47,AL1053,Alabama,shall issue (permit not required),Modify,2023,1,1


Registros encontrados: 12
